# import Libraries 

In [47]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain.agents import Tool, initialize_agent, load_tools


In [48]:
from langchain_groq import ChatGroq
from IPython.display import Markdown, display

In [49]:
load_dotenv()
GROQ_API = os.getenv("GROQ_API_KEY")
model_name = 'llama-3.3-70b-versatile'


llm  =  ChatGroq(api_key=GROQ_API,
    model_name=model_name,
    temperature=0.0,
    
)

In [50]:
tools = load_tools(
    ['llm-math'],
    llm=llm
)

print(tools[0].name, tools[0].description)

Calculator Useful for when you need to answer questions about math.


In [38]:
agent = initialize_agent(
    agent="zero-shot-react-description",
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=3 # to avoid high bills from the LLM
)


## ReAct framework = Reasoning and Action

## ReAct Methodology
The chatbot employs a step-by-step reasoning approach to interpret and respond to user prompts:
1. Thought: Analyze the prompt internally.
2. Action: Execute an action, such as calling a tool (SQL or image processing).
3. Observation: Incorporate the tool’s output into subsequent reasoning steps.
4. Answer: Provide the final response. 


In [39]:
print(agent.agent.llm_chain.prompt.template)

Answer the following questions as best you can. You have access to the following tools:

Calculator(*args: Any, callbacks: Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType] = None, tags: Optional[list[str]] = None, metadata: Optional[dict[str, Any]] = None, **kwargs: Any) -> Any - Useful for when you need to answer questions about math.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [Calculator]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [40]:
query = "If James is currently 45 years old, how old will he be in 50 years? \
    If he has 4 kids and adopted 7 more, how many children does he have?"
result = agent.invoke(query)
print(result['output'])



> Entering new AgentExecutor chain...
Thought: To find out how old James will be in 50 years, we need to add 50 to his current age. We also need to calculate the total number of children he has by adding the number of kids he has to the number of kids he adopted.

Action: Calculator
Action Input: 45 + 50
Observation: Answer: 95
Thought:Thought: Now that we know James will be 95 years old in 50 years, we need to calculate the total number of children he has. He has 4 kids and adopted 7 more, so we need to add these two numbers together.

Action: Calculator
Action Input: 4 + 7
Observation: Answer: 11
Thought:Thought: I now know the final answer
Final Answer: In 50 years, James will be 95 years old and he has a total of 11 children.

> Finished chain.
In 50 years, James will be 95 years old and he has a total of 11 children.


In [41]:

#display(Markdown(f" ChatGPT ::: {llm.predict('what is 3.1^2.1')}"))
print(f" ChatGPT ::: {llm.predict('what is 3.1^2.1')}")





 ChatGPT ::: To calculate 3.1^2.1, we need to use the power operation, which is exponentiation.

3.1^2.1 = 3.1 to the power of 2.1

Using a calculator or a mathematical library, we get:

3.1^2.1 ≈ 9.6161

So, 3.1^2.1 is approximately 9.62 (rounded to two decimal places).


In [42]:
result = agent.invoke('what is 3.1^2.1')
print(result['output'])



> Entering new AgentExecutor chain...
To find the value of 3.1^2.1, I need to calculate the result of raising 3.1 to the power of 2.1.

Action: Calculator
Action Input: 3.1^2.1
Observation: Answer: 10.761171606099687
Thought:Thought: I now know the final answer
Final Answer: 10.761171606099687

> Finished chain.
10.761171606099687


In [43]:
from langchain import Wikipedia
from langchain.agents.react.base import DocstoreExplorer

In [44]:
docstore = DocstoreExplorer(Wikipedia())
tools = [
    Tool(
        name="Search",
        func=docstore.search,
        description="search wikipedia"
    ),
    Tool(
        name="Lookup",
        func=docstore.lookup,
        description="lookup a term in wikipedia"
    )
]


In [45]:
# initialize our agent
docstore_agent = initialize_agent(
    tools,
    llm,
    agent="react-docstore",
    verbose=True,
    max_iterations=10
)

In [46]:
query = "What were Einstein's main beliefs?"
result = docstore_agent.invoke(query)



> Entering new AgentExecutor chain...
To answer this question, I'll need to search for Albert Einstein's main beliefs.

Action: Search[Albert Einstein's beliefs]


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
print(docstore_agent.agent.llm_chain.prompt.template)

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv
import openai
from langchain.llms import OpenAI
from langchain.agents import Tool, initialize_agent, load_tools

from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory

In [ ]:
# memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
# Second Generic Tool
prompt = PromptTemplate(
    input_variables=["query"],
    template="{query}"
)

In [ ]:
llm_chain = prompt | llm

In [ ]:
# Initialize the LLM Tool
llm_tool = Tool(
    name="Language Model",
    func=llm_chain.invoke,
    description="Use this tool for general queries and logic"
)
 
tools = load_tools(
    ['llm-math'],
    llm=llm
)

In [ ]:
tools.append(llm_tool) # adding the new tool to our tools list


In [ ]:
# Conversational Agent
conversational_agent = initialize_agent(
    agent="conversational-react-description",
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=4,
    memory=memory
)

In [ ]:
query = "How old is a person born in 1917 in 2026"

In [ ]:
print(conversational_agent.agent.llm_chain.prompt.template)

In [ ]:
result = conversational_agent(query)
print(result['output'])


In [ ]:
query_two = "How old would that person be if their age is multiplied by 100 and add resulte this 159*857?"

In [ ]:
results = conversational_agent(query_two)
print(results['output'])